# Feature engineering - advanced data preparation pipeline  | Sebislaw

## Libraries

In [65]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier

import xgboost as xgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
# from tabpfn import TabPFNClassifier

## Data

In [66]:
data_path = '..\\..\\..\\data'
pd.set_option('display.max_columns', None)

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

## Data preparation pipeline

In [67]:
def prepare_data(df):
        
    """
    This function duplicates and flips a game record.
    Now two records with the same data are present, 
    but viewed from perspectives of two different teams.
    """
    
    df = df[[
         'Season', 'DayNum', 'NumOT',
         'WTeamID',  'WScore', 'WLoc',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
         'LTeamID', 'LScore',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'
    ]]
    dfswap = df[[
         'Season', 'DayNum', 'NumOT',
         'LTeamID', 'LScore', 'WLoc',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF',
         'WTeamID',  'WScore',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
    ]].copy()
    
    dfswap.loc[df['WLoc'] == 'H', 'WLoc'] = 'A'
    dfswap.loc[df['WLoc'] == 'A', 'WLoc'] = 'H'
        
    df = df.rename(columns={'WLoc': 'location'})
    dfswap = dfswap.rename(columns={'WLoc': 'location'})
        
    df.columns = [x.replace('W','T1_').replace('L','T2_') for x in list(df.columns)]
    dfswap.columns = [x.replace('L','T1_').replace('W','T2_') for x in list(dfswap.columns)]
    
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output.loc[output.location=='N','location'] = '0'
    output.loc[output.location=='H','location'] = '1'
    output.loc[output.location=='A','location'] = '-1'
    output.location = output.location.astype(int)
        
    output['PointDiff'] = output['T1_Score'] - output['T2_Score']
    
    return output

def get_data(regular_results, tourney_results, seeds, prepared=False, location_multiplier=[1, 1], win_ratio_days_back=14):

    """
    This function uses the prepare_data function in order to create
    a data frame with season statistics for each team.
    These statistics are added to records with games played in
    tournament to make data 'x' used in model to predict the game 
    result 'y'. The output is a data frame that contains data 'x'
    and also label 'y' can be easily calculated based on score difference in matches.
    """

    if prepared:
        regular_data = regular_results.copy()
        tourney_data = tourney_results.copy()
    else:
        # make data frames with extra rows to represent the perspective of losing team
        regular_data = prepare_data(regular_results)
        tourney_data = prepare_data(tourney_results)

    # ----------------------------------- Add reward/penalty for playing in home or away
    if location_multiplier[0] == 1 and location_multiplier[1] == 1:
        # data frame with mean game statistics for a given team in a given season
        season_statistics  = regular_data.groupby(["Season", 'T1_TeamID'])[
            [
                'T1_Score', 'T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA',
                'T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF',
                'T2_Score', 'T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA',
                'T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF',
                'PointDiff'
            ]
        ].agg('mean').reset_index()
    else:
        # Define which columns to adjust (you can add or remove columns as needed)
        T1_cols = ['T1_Score','T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA','T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF']
        T2_cols = ['T2_Score','T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA','T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF']
    
        # Convert the relevant columns to float before applying the adjustment function.
        cols_to_float = T1_cols + T2_cols
        regular_data[cols_to_float] = regular_data[cols_to_float].astype(float)
        
        def adjust_stats(row):
            # Determine multipliers based on location
            if row['location'] == 1:
                factor_T1 = location_multiplier[0]  # penalize Team1 stats (home)
                factor_T2 = location_multiplier[1]  # boost Team2 stats
            elif row['location'] == -1:
                factor_T1 = location_multiplier[1]  # boost Team1 stats (away)
                factor_T2 = location_multiplier[0]  # penalize Team2 stats
            else:
                factor_T1 = 1.0
                factor_T2 = 1.0
        
            # Adjust Team1 stats
            for col in T1_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T1
        
            # Adjust Team2 stats
            for col in T2_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T2
        
            # Recalculate derived statistics (if needed)
            if 'T1_Score' in row and 'T2_Score' in row:
                row['PointDiff'] = row['T1_Score'] - row['T2_Score']
            return row
        
        # Apply the adjustment function row-wise.
        regular_data_adjusted = regular_data.apply(adjust_stats, axis=1)
        
        # Now group by Season and T1_TeamID to compute season averages for the adjusted statistics.
        stats_columns = T1_cols[1:] + T2_cols[1:] + ['PointDiff']  # Exclude T1_TeamID from stats if present.
        season_statistics = regular_data_adjusted.groupby(["Season", 'T1_TeamID'])[stats_columns].agg('mean').reset_index()
    # -----------------------------------
    
    # mean statistics for team and team's opponent's
    season_statistics_T1 = season_statistics.copy()
    season_statistics_T2 = season_statistics.copy()
    
    season_statistics_T1.columns = ["T1_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T1.columns)]
    season_statistics_T2.columns = ["T2_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T2.columns)]
    season_statistics_T1.columns.values[0] = "Season"
    season_statistics_T2.columns.values[0] = "Season"
    season_statistics_T1 = season_statistics_T1.rename(columns={'T1_Score': 'T1_Score_mean'})
    season_statistics_T2 = season_statistics_T2.rename(columns={'T2_Score': 'T2_Score_mean'})
    
    # data frame containing game's result
    tourney_data = tourney_data[['Season', 'DayNum', 'T1_TeamID', 'T1_Score', 'T2_TeamID' ,'T2_Score', 'location']]
    tourney_data = pd.merge(tourney_data, season_statistics_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, season_statistics_T2, on = ['Season', 'T2_TeamID'], how = 'left')

    calculate_win_ratio_days_back = 132 - win_ratio_days_back
    
    # data frame with win fraction from last x days for a given team in a given season
    last14days_stats_T1 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T1['win'] = np.where(last14days_stats_T1['PointDiff']>0,1,0)
    last14days_stats_T1 = last14days_stats_T1.groupby(['Season','T1_TeamID'])['win'].mean().reset_index(name='T1_win_ratio_14d')
    
    last14days_stats_T2 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T2['win'] = np.where(last14days_stats_T2['PointDiff']<0,1,0)
    last14days_stats_T2 = last14days_stats_T2.groupby(['Season','T2_TeamID'])['win'].mean().reset_index(name='T2_win_ratio_14d')
    
    # add to tourney_data column with win fraction for winning and losing team
    tourney_data = pd.merge(tourney_data, last14days_stats_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, last14days_stats_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # get seeds with no regional division
    seeds['seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))
    
    # give each team a raw seed
    seeds_T1 = seeds[['Season','TeamID','seed']].copy()
    seeds_T2 = seeds[['Season','TeamID','seed']].copy()
    seeds_T1.columns = ['Season','T1_TeamID','T1_seed']
    seeds_T2.columns = ['Season','T2_TeamID','T2_seed']
    
    # add seeds to turney data for team 1 and team 2
    tourney_data = pd.merge(tourney_data, seeds_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, seeds_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # add a seed difference column
    tourney_data["Seed_diff"] = tourney_data["T1_seed"] - tourney_data["T2_seed"]

    return tourney_data

def get_df(seeds,
            season_games, season_range, days_back,
            tourney_games, tourney_range, 
            location_multiplier=[1, 1],
           win_ratio_days_back=14):

    """
    This function uses get_data function to get a data
    frame which is then used to make 'x' and 'y' data
    used in models.
    """
    
    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]
    
    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Get final data frame
    df = get_data(regular_results, tourney_results, seeds, location_multiplier=[1, 1], win_ratio_days_back=win_ratio_days_back)
    
    return df

def get_final_df(seeds,
                   season_games, season_range, days_back,
                   tourney_games, tourney_range, 
                   SampleSubmissionStage1,
                  location_multiplier=[1, 1],
                win_ratio_days_back=14):

    """
    This function works the same as function get_x_y,
    but also creates (at the moment it's the same as get_x_y)
    aditional data points mostly with NaN values that matches
    the submission format (parsed team matchups with the sample submission file).
    """

    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]

    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Assume sample_submission is a DataFrame with an "ID" column like "2023_1101_1102"
    # and tourney_results is a DataFrame with columns including: Season, WTeamID, LTeamID, DayNum, WScore, LScore, WLoc, etc.
    # Filter rows where the ID starts with the specified season (followed by an underscore)
    final_season = tourney_range[0]
    sample_submission_copy = SampleSubmissionStage1.copy()
    sample_submission = sample_submission_copy[sample_submission_copy['ID'].str.startswith(f"{final_season}_")]
    sample_submission = sample_submission.drop(columns=['Pred'])
    
    sample_submission[['Season', 'Team1', 'Team2']] = sample_submission['ID'].str.split('_', expand=True)
    sample_submission['Season'] = sample_submission['Season'].astype(float)
    sample_submission['Team1'] = sample_submission['Team1'].astype(float)
    sample_submission['Team2'] = sample_submission['Team2'].astype(float)
    
    regular_data_final = prepare_data(regular_results)
    tourney_data_final  = prepare_data(tourney_results)
    
    tourney_data =  get_data(regular_data_final, tourney_data_final,
                             seeds, prepared=True,
                          win_ratio_days_back=win_ratio_days_back)
    tourney_data = pd.merge(
        sample_submission,
        tourney_data,
        left_on=['Season', 'Team1', 'Team2'],
        right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
        how='left'
    )
    tourney_data['T1_TeamID'] = tourney_data['Team1']
    tourney_data['T2_TeamID'] = tourney_data['Team2']
    tourney_data = tourney_data.drop(['ID', 'Team1', 'Team2'], axis=1)
    
    return tourney_data

def correct_predictions_based_on_seed(x, y, maximum_favoured_seed = 4, number_of_added_columns=1):
    """
    Sets the winnning chance to 1 or 0 based on the seed difference.
    """
    for i in range(len(x)):
        if x[i][-1-number_of_added_columns] <= (-16 + maximum_favoured_seed * 2 - 1):
            y[i] = 1
        elif x[i][-1-number_of_added_columns] >= (16 - maximum_favoured_seed * 2 + 1):
            y[i] = 0
    return y

def clear_na_from_x_y(x, y):
    """
    The data frame for final season is in format matching the submission file.
    This function clears NaNs from data.
    """
    # Create masks for training data:
    mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
    x_clean = x[mask_train]
    y_clean = y[mask_train]
    return x_clean, y_clean

def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[7:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

def get_all_core_data(
    regular_results, tourney_results, seeds, SampleSubmissionStage1,
    final_season = 2024, # the season we want to predict, so for out submission it will be 2025
    start_season = 2005, # from which ponit should we begin creating data
    season_years_list  = [[i-1, i] for i in range(2005, 2024+1)], # at which seasons to look at when calculating team's stats
    days_back = 15, # how many days back from the start of tourney to calculate team's stats per season
    maximum_favoured_seed = 4, # Set the predicted probability of winning to 1 for seeds <= maximum_favoured_seed and to 0 for >= 16-maximum_favoured_seed
    location_multiplier=[0.95, 1.05], # home penalty, away bonus 
    include_men = True, # include M... data sets when preparing x and y
    include_women = True, # include W... data sets when preparing x and y
    win_ratio_days_back = 14):

    """
    The idea of this function is to easily get data needed to train and test the model later on, with minimal code
    to not clutter the netebook.
    
    This function outputs df, x, y, df_final, x_final_season, y_final_season.
    
    df is a data frame with first 6 columns from tourney games and other calculated from other data frames.
    
    Adding a column to df and executing x_y_from_data_frame(df) function will yield x with added data.
    
    Note that df_final has the same structure as df, but also with rows with NaNs. The rows with missing information are there
    to match the sumbission file format. The separation of those data frames is to ensure that information from last season doesn't
    leak into training data due to poorly written code.
    
    x has df columns from location onwardsthe columns before that are from tourney games and are used to calculate y (based on points).
    
    y has label 0 or 1 (lose or win) and nan if the correspoinding data in x was nan.
    
    There is also x_final_season and y_final_season aquired from df_final, which are the same as x and y, but like df_final, they have NaNs. 
    """

    # This ensures that we simulate the scenario in competition
    tourney_results_final = tourney_results[tourney_results['Season'] == final_season]
    tourney_results = tourney_results[tourney_results['Season'] < final_season]
    regular_results = regular_results[regular_results['Season'] != 2020] # This year had no tournament data
    tourney_years_list = [[i] for i in range(start_season, final_season+1)]
    
    # Arrays to store data
    data = []
    data_final = []
    for season_years, tourney_years in zip(season_years_list, tourney_years_list):
            
        # Create data separately for the of games
        # The separation is to ensure there is no data leak
        if tourney_years[0] == final_season:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results_final, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data_final.append(data_tmp)
        else:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data.append(data_tmp)
            
    df = pd.concat(data, ignore_index=True)
    df_final = pd.concat(data_final, ignore_index=True)
    
    return df, df_final

def brier_for_all_years(year_range, season_years_list):
    
    df_train_women_list = []
    df_test_women_list = []
    df_train_men_list = []
    df_test_men_list = []
    
    for year, season_years in zip(year_range, season_years_list):
    
        final_season = year

        for sex in ['woman', 'man']:
            
            if sex == 'woman':
                include_men = False
                include_women = True
            elif sex == 'man':
                include_men = True
                include_women = False

            # ----------------------------------------------------------
            # READ DATA
            regular_results = pd.concat([
                MRegularSeasonDetailedResults.copy() if include_men else None,
                WRegularSeasonDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            tourney_results = pd.concat([
                MNCAATourneyDetailedResults.copy() if include_men else None,
                WNCAATourneyDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            seeds = pd.concat([
                MNCAATourneySeeds.copy() if include_men else None,
                WNCAATourneySeeds.copy() if include_women else None
            ], ignore_index=True)
            # ----------------------------------------------------------
            # GET ALL DATA NEEDED TO USE THE MODELS
            df_train, df_test = get_all_core_data(
                regular_results, tourney_results, seeds, SampleSubmissionStage1, final_season = final_season,
                start_season = start_season, season_years_list  = season_years, days_back = days_back,
                maximum_favoured_seed = maximum_favoured_seed, location_multiplier=location_multiplier,
                include_men = include_men, include_women = include_women, win_ratio_days_back = win_ratio_days_back)
            # ----------------------------------------------------------
            # ADD TEAM AND COACH ELO AND REPLACE NAN WITH MEAN
            elo = pd.read_csv(join(data_path, 'elo.csv'))
            elo['CoachELO'] = elo['CoachELO'].fillna(elo['CoachELO'].mean())
            elo = elo.drop(['CoachName'], axis=1)
            def add_elo_column(df):
                df = df.copy()
                df = pd.merge(
                        df,
                        elo[['Season', 'DayNum', 'TeamID', 'TeamELO', 'CoachELO']],
                        left_on=['Season', 'DayNum', 'T1_TeamID'],
                        right_on=['Season', 'DayNum', 'TeamID'],
                        how='left'
                    )
                df = df.drop(['TeamID'], axis=1)
                return df
            df_train = add_elo_column(df_train)
            df_test = add_elo_column(df_test)
            # ----------------------------------------------------------
            if sex == 'woman':
                df_train_women_list.append(df_train.copy())
                df_test_women_list.append(df_test.copy())
            elif sex == 'man':
                df_train_men_list.append(df_train.copy())
                df_test_men_list.append(df_test.copy())
                
        for i in range(len(df_train_men_list)):
            df_train_women_list[i] = df_train_women_list[i][list(df_train_men_list[0])]
            df_test_women_list[i] = df_test_women_list[i][list(df_train_men_list[0])]
            
    return df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list

## Get data to use models on

In [62]:
columns_to_include_women = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]
columns_to_include_men = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]

year_range = [2022]
start_season = 2010 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 25 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.95, 1.05] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
# -------------------------------------------
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
for i in range(len(year_range)):

    if len(year_range) > 1:
        df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
        df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
        df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
        df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
        
        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())
        
        x_train_women_list.append(x_train_women)
        x_test_women_list.append(x_test_women)
        x_train_men_list.append(x_train_men)
        x_test_men_list.append(x_test_men)

        y_train_women_list.append(y_train_women)
        y_test_women_list.append(y_test_women)
        y_train_men_list.append(y_train_men)
        y_test_men_list.append(y_test_men)
    
    else:
        df_train_women_list = df_train_women_list[0][columns_to_include_women]
        df_test_women_list = df_test_women_list[0][columns_to_include_women]
        df_train_men_list = df_train_men_list[0][columns_to_include_men]
        df_test_men_list = df_test_men_list[0][columns_to_include_men]

        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list)
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list)
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list)
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list)

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())

# Training models

## Logistic regression

In [63]:
import optuna
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import StratifiedKFold

# Define an objective function that uses StratifiedKFold CV.
def objective_lr_cv(trial, X, y, n_folds=5):  # Increased folds for better generalization
    # Stronger regularization: Reduce range of C to prevent overfitting
    C = trial.suggest_float("C", 1e-4, 10, log=True)  # Lower upper bound

    solver_men = "liblinear" if best_params_men["penalty"] == "l1" else "lbfgs"
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    solver = solver_men  # More stable for both L1 and L2

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X, y):
        X_train_cv, X_val_cv = X[train_idx], X[val_idx]
        y_train_cv, y_val_cv = y[train_idx], y[val_idx]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(penalty=penalty, C=C, solver=solver, max_iter=3000))  # Increased max_iter
        ])

        model.fit(X_train_cv, y_train_cv)
        y_pred = model.predict_proba(X_val_cv)[:, 1]
        scores.append(brier_score_loss(y_val_cv, y_pred))

    return np.mean(scores)

###########################
# For Women
###########################
study_women = optuna.create_study(direction="minimize")
study_women.optimize(lambda trial: objective_lr_cv(trial, x_train_women, y_train_women), n_trials=100)  # Increased trials for better tuning

best_params_women = study_women.best_params
print("Best Logistic Regression CV Params (Women):", best_params_women)

best_lr_women = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(penalty=best_params_women["penalty"],
                                       C=best_params_women["C"],
                                       solver="saga",
                                       max_iter=3000))
])
best_lr_women.fit(x_train_women, y_train_women)
y_pred_women_lr = best_lr_women.predict_proba(x_test_women)[:, 1]
brier_women_lr = brier_score_loss(y_test_women, y_pred_women_lr)
print("Final Logistic Regression Brier Score (Women):", brier_women_lr)

###########################
# For Men
###########################
study_men = optuna.create_study(direction="minimize")
study_men.optimize(lambda trial: objective_lr_cv(trial, x_train_men, y_train_men), n_trials=100)

best_params_men = study_men.best_params
print("Best Logistic Regression CV Params (Men):", best_params_men)

best_lr_men = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(penalty=best_params_men["penalty"],
                                       C=best_params_men["C"],
                                       solver="saga",
                                       max_iter=3000))
])
best_lr_men.fit(x_train_men, y_train_men)
y_pred_men_lr = best_lr_men.predict_proba(x_test_men)[:, 1]
brier_men_lr = brier_score_loss(y_test_men, y_pred_men_lr)
print("Final Logistic Regression Brier Score (Men):", brier_men_lr)


[I 2025-03-20 00:02:33,399] A new study created in memory with name: no-name-9c300783-4ea0-4333-b586-19e1fefc6a70
[I 2025-03-20 00:02:33,434] Trial 0 finished with value: 0.16363996838224731 and parameters: {'C': 0.00822216636024389, 'penalty': 'l1'}. Best is trial 0 with value: 0.16363996838224731.
[I 2025-03-20 00:02:33,481] Trial 1 finished with value: 0.1514585381671772 and parameters: {'C': 0.021769859431145833, 'penalty': 'l1'}. Best is trial 1 with value: 0.1514585381671772.
[I 2025-03-20 00:02:33,531] Trial 2 finished with value: 0.1499136792218823 and parameters: {'C': 0.006680590087406687, 'penalty': 'l2'}. Best is trial 2 with value: 0.1499136792218823.
[I 2025-03-20 00:02:33,564] Trial 3 finished with value: 0.22688612553710968 and parameters: {'C': 0.0033175279338872616, 'penalty': 'l1'}. Best is trial 2 with value: 0.1499136792218823.
[I 2025-03-20 00:02:33,614] Trial 4 finished with value: 0.1453709108071595 and parameters: {'C': 0.05106271266038067, 'penalty': 'l1'}. Be

[I 2025-03-20 00:02:39,136] Trial 44 finished with value: 0.14438326066111234 and parameters: {'C': 0.08561727918530622, 'penalty': 'l1'}. Best is trial 21 with value: 0.14437177349849234.
[I 2025-03-20 00:02:39,211] Trial 45 finished with value: 0.1443736926325218 and parameters: {'C': 0.09123067271394258, 'penalty': 'l1'}. Best is trial 21 with value: 0.14437177349849234.
[I 2025-03-20 00:02:39,336] Trial 46 finished with value: 0.1458467654955827 and parameters: {'C': 0.23237678164813144, 'penalty': 'l1'}. Best is trial 21 with value: 0.14437177349849234.
[I 2025-03-20 00:02:39,452] Trial 47 finished with value: 0.14718168425980127 and parameters: {'C': 0.03702629434191447, 'penalty': 'l1'}. Best is trial 21 with value: 0.14437177349849234.
[I 2025-03-20 00:02:39,643] Trial 48 finished with value: 0.14717074038746414 and parameters: {'C': 0.4431972811583075, 'penalty': 'l1'}. Best is trial 21 with value: 0.14437177349849234.
[I 2025-03-20 00:02:39,727] Trial 49 finished with value: 

[I 2025-03-20 00:02:43,067] Trial 88 finished with value: 0.15164230246885382 and parameters: {'C': 0.02118351885247988, 'penalty': 'l1'}. Best is trial 51 with value: 0.14436803877188578.
[I 2025-03-20 00:02:43,200] Trial 89 finished with value: 0.1464114800473421 and parameters: {'C': 0.2961833034443037, 'penalty': 'l1'}. Best is trial 51 with value: 0.14436803877188578.
[I 2025-03-20 00:02:43,278] Trial 90 finished with value: 0.14438276176695491 and parameters: {'C': 0.09500114172972177, 'penalty': 'l1'}. Best is trial 51 with value: 0.14436803877188578.
[I 2025-03-20 00:02:43,358] Trial 91 finished with value: 0.14474458024144707 and parameters: {'C': 0.06289310067405006, 'penalty': 'l1'}. Best is trial 51 with value: 0.14436803877188578.
[I 2025-03-20 00:02:43,436] Trial 92 finished with value: 0.14437648482777854 and parameters: {'C': 0.09252709785764116, 'penalty': 'l1'}. Best is trial 51 with value: 0.14436803877188578.
[I 2025-03-20 00:02:43,535] Trial 93 finished with value:

Best Logistic Regression CV Params (Women): {'C': 0.09037485057111518, 'penalty': 'l1'}
Final Logistic Regression Brier Score (Women): 0.1644912050964949


[I 2025-03-20 00:02:44,217] Trial 0 finished with value: 0.2040357495044857 and parameters: {'C': 0.5901555227540349, 'penalty': 'l2'}. Best is trial 0 with value: 0.2040357495044857.
[I 2025-03-20 00:02:45,044] Trial 1 finished with value: 0.20476973214404168 and parameters: {'C': 5.9874627779743435, 'penalty': 'l1'}. Best is trial 0 with value: 0.2040357495044857.
[I 2025-03-20 00:02:45,131] Trial 2 finished with value: 0.20260710063619541 and parameters: {'C': 0.12079387938795283, 'penalty': 'l2'}. Best is trial 2 with value: 0.20260710063619541.
[I 2025-03-20 00:02:45,184] Trial 3 finished with value: 0.2030050037199982 and parameters: {'C': 0.004697790671390097, 'penalty': 'l2'}. Best is trial 2 with value: 0.20260710063619541.
[I 2025-03-20 00:02:45,289] Trial 4 finished with value: 0.20483373495668777 and parameters: {'C': 3.1822066848353043, 'penalty': 'l2'}. Best is trial 2 with value: 0.20260710063619541.
[I 2025-03-20 00:02:45,350] Trial 5 finished with value: 0.204099535035

[I 2025-03-20 00:02:49,979] Trial 44 finished with value: 0.20210376229334762 and parameters: {'C': 0.02092568926137872, 'penalty': 'l1'}. Best is trial 23 with value: 0.20019935204726433.
[I 2025-03-20 00:02:50,029] Trial 45 finished with value: 0.21751583171025696 and parameters: {'C': 0.006508873417046179, 'penalty': 'l1'}. Best is trial 23 with value: 0.20019935204726433.
[I 2025-03-20 00:02:50,071] Trial 46 finished with value: 0.25 and parameters: {'C': 0.00013442451206791353, 'penalty': 'l1'}. Best is trial 23 with value: 0.20019935204726433.
[I 2025-03-20 00:02:50,129] Trial 47 finished with value: 0.20409602806539154 and parameters: {'C': 0.014471520263161692, 'penalty': 'l1'}. Best is trial 23 with value: 0.20019935204726433.
[I 2025-03-20 00:02:50,202] Trial 48 finished with value: 0.2002430152265032 and parameters: {'C': 0.040148048151676004, 'penalty': 'l1'}. Best is trial 23 with value: 0.20019935204726433.
[I 2025-03-20 00:02:50,329] Trial 49 finished with value: 0.20122

[I 2025-03-20 00:02:54,868] Trial 88 finished with value: 0.20029481189903114 and parameters: {'C': 0.07494849158779926, 'penalty': 'l1'}. Best is trial 65 with value: 0.2001846307802678.
[I 2025-03-20 00:02:54,927] Trial 89 finished with value: 0.20456038191658127 and parameters: {'C': 0.013674654244189682, 'penalty': 'l1'}. Best is trial 65 with value: 0.2001846307802678.
[I 2025-03-20 00:02:55,043] Trial 90 finished with value: 0.20113525189590478 and parameters: {'C': 0.13403816859785478, 'penalty': 'l1'}. Best is trial 65 with value: 0.2001846307802678.
[I 2025-03-20 00:02:55,159] Trial 91 finished with value: 0.20020295606612257 and parameters: {'C': 0.06015952605816391, 'penalty': 'l1'}. Best is trial 65 with value: 0.2001846307802678.
[I 2025-03-20 00:02:55,243] Trial 92 finished with value: 0.20024340993374637 and parameters: {'C': 0.04553914718644791, 'penalty': 'l1'}. Best is trial 65 with value: 0.2001846307802678.
[I 2025-03-20 00:02:55,343] Trial 93 finished with value: 0

Best Logistic Regression CV Params (Men): {'C': 0.06297569603469304, 'penalty': 'l1'}
Final Logistic Regression Brier Score (Men): 0.2096146756014256


## Testing the model

In [ ]:
year_range = [i for i in range(2011, 2020)] + [i for i in range(2022, 2025)]
start_season = 2010 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 25 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.95, 1.05] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
# ----------------------------------------------------------
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
brier_women_list = []
brier_men_list = []
brier_all_list = []
for i in range(len(year_range)):
    df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
    df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
    df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
    df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
    
    x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
    x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
    x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
    x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])
    
    x_train_women_list.append(x_train_women)
    x_test_women_list.append(x_test_women)
    x_train_men_list.append(x_train_men)
    x_test_men_list.append(x_test_men)
    
    y_train_women_list.append(y_train_women)
    y_test_women_list.append(y_test_women)
    y_train_men_list.append(y_train_men)
    y_test_men_list.append(y_test_men)
    
# -------------------------------------------------------

for i in range(len(year_range)):
    print("Predictions for season ", year_range[i])
    x_train_women = x_train_women_list[i]
    y_train_women = y_train_women_list[i]
    x_test_women = x_test_women_list[i]
    y_test_women = y_test_women_list[i]
    
    best_lr_women.fit(x_train_women, y_train_women)
    y_prob_women = best_lr_women.predict_proba(x_test_women)[:, 1]

    score = brier_score_loss(y_test_women, y_prob_women)
    brier_women_list.append(score)
    print('LogisticRegressionCV for women trained on women', score)
    # ----------------------------------------------
    x_train_men = x_train_men_list[i]
    y_train_men = y_train_men_list[i]
    x_test_men = x_test_men_list[i]
    y_test_men = y_test_men_list[i]
    
    best_lr_men.fit(x_train_men, y_train_men)
    y_prob_men = best_lr_men.predict_proba(x_test_men)[:, 1]
    
    score = brier_score_loss(y_test_men, y_prob_men)
    brier_men_list.append(score)
    print('LogisticRegressionCV for men trained on men', score)
    # ----------------------------------------------
    y_prob = np.concatenate((y_prob_women, y_prob_men))
    y_test = np.concatenate((y_test_women, y_test_men))
    score = brier_score_loss(y_test, y_prob)
    brier_all_list.append(score)
    print('LogisticRegressionCV for all trained on women and men separately', score, '<---')
    print()
    # ----------------------------------------------
print()
print('The mean score when trained on women and tested on women was: ', np.mean(brier_women_list), 
      'with a std of ', np.std(brier_women_list))
print('The mean score when trained on men and tested on men was: ', np.mean(brier_men_list), 
      'with a std of ', np.std(brier_men_list))
print('The mean score when trained separately and tested on both was: ', np.mean(brier_all_list), 
      'with a std of ', np.std(brier_all_list))

# No more separation, all columns

In [76]:
columns_to_include_women = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
 'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
 'T1_FTM',
 'T1_FTA',
 'T1_OR',
 'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
 'T1_Blk',
 'T1_PF',
                      
 'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
 'T1_opponent_FTM',
 'T1_opponent_FTA',
 'T1_opponent_OR',
 'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
 'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
 'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
 'T2_FTM',
 'T2_FTA',
 'T2_OR',
 'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
 'T2_Blk',
 'T2_PF',
                      
 'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
 'T2_opponent_FTM',
 'T2_opponent_FTA',
 'T2_opponent_OR',
 'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
 'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]
columns_to_include_men = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
 'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
 'T1_FTM',
 'T1_FTA',
 'T1_OR',
 'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
 'T1_Blk',
 'T1_PF',
                      
 'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
 'T1_opponent_FTM',
 'T1_opponent_FTA',
 'T1_opponent_OR',
 'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
 'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
 'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
 'T2_FTM',
 'T2_FTA',
 'T2_OR',
 'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
 'T2_Blk',
 'T2_PF',
                      
 'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
 'T2_opponent_FTM',
 'T2_opponent_FTA',
 'T2_opponent_OR',
 'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
 'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]

year_range = [2022]
start_season = 2010 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 25 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.95, 1.05] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
# -------------------------------------------
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
for i in range(len(year_range)):

    if len(year_range) > 1:
        df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
        df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
        df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
        df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
        
        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())
        
        x_train_women_list.append(x_train_women)
        x_test_women_list.append(x_test_women)
        x_train_men_list.append(x_train_men)
        x_test_men_list.append(x_test_men)

        y_train_women_list.append(y_train_women)
        y_test_women_list.append(y_test_women)
        y_train_men_list.append(y_train_men)
        y_test_men_list.append(y_test_men)
    
    else:
        df_train_women_list = df_train_women_list[0][columns_to_include_women]
        df_test_women_list = df_test_women_list[0][columns_to_include_women]
        df_train_men_list = df_train_men_list[0][columns_to_include_men]
        df_test_men_list = df_test_men_list[0][columns_to_include_men]

        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list)
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list)
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list)
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list)

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())

In [77]:
import optuna
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import StratifiedKFold

# Define an objective function that uses StratifiedKFold CV.
def objective_lr_cv(trial, X, y, n_folds=5):  # Increased folds for better generalization
    # Stronger regularization: Reduce range of C to prevent overfitting
    C = trial.suggest_float("C", 1e-4, 10, log=True)  # Lower upper bound

    solver_men = "liblinear" if best_params_men["penalty"] == "l1" else "lbfgs"
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    solver = solver_men  # More stable for both L1 and L2

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X, y):
        X_train_cv, X_val_cv = X[train_idx], X[val_idx]
        y_train_cv, y_val_cv = y[train_idx], y[val_idx]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(penalty=penalty, C=C, solver=solver, max_iter=3000))  # Increased max_iter
        ])

        model.fit(X_train_cv, y_train_cv)
        y_pred = model.predict_proba(X_val_cv)[:, 1]
        scores.append(brier_score_loss(y_val_cv, y_pred))

    return np.mean(scores)

###########################
# For Women
###########################
x_train_women = np.concatenate((x_train_women, x_train_men))
y_train_women = np.concatenate((y_train_women, y_train_men))

study_women = optuna.create_study(direction="minimize")
study_women.optimize(lambda trial: objective_lr_cv(trial, x_train_women, y_train_women), n_trials=100)  # Increased trials for better tuning

best_params_women = study_women.best_params
print("Best Logistic Regression CV Params (All):", best_params_women)

best_lr_women = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(penalty=best_params_women["penalty"],
                                       C=best_params_women["C"],
                                       solver="saga",
                                       max_iter=3000))
])
best_lr_women.fit(x_train_women, y_train_women)
y_pred_women_lr = best_lr_women.predict_proba(x_test_women)[:, 1]
brier_women_lr = brier_score_loss(y_test_women, y_pred_women_lr)
print("Final Logistic Regression Brier Score (All):", brier_women_lr)

[I 2025-03-20 01:07:55,606] A new study created in memory with name: no-name-ed525a3e-4361-4d1c-be65-f7a33de81252
[I 2025-03-20 01:07:55,704] Trial 0 finished with value: 0.24208336709472356 and parameters: {'C': 0.0017439174474762921, 'penalty': 'l1'}. Best is trial 0 with value: 0.24208336709472356.
[I 2025-03-20 01:07:55,816] Trial 1 finished with value: 0.25 and parameters: {'C': 0.00028014889270815206, 'penalty': 'l1'}. Best is trial 0 with value: 0.24208336709472356.
[I 2025-03-20 01:07:56,450] Trial 2 finished with value: 0.17537030500087436 and parameters: {'C': 0.3221270960849997, 'penalty': 'l1'}. Best is trial 2 with value: 0.17537030500087436.
[I 2025-03-20 01:07:56,563] Trial 3 finished with value: 0.2235870657099996 and parameters: {'C': 0.0001327991274000699, 'penalty': 'l2'}. Best is trial 2 with value: 0.17537030500087436.
[I 2025-03-20 01:07:56,707] Trial 4 finished with value: 0.25 and parameters: {'C': 0.0008824994993570505, 'penalty': 'l1'}. Best is trial 2 with va

[I 2025-03-20 01:08:16,753] Trial 44 finished with value: 0.17472181450309643 and parameters: {'C': 0.07752016643722703, 'penalty': 'l1'}. Best is trial 14 with value: 0.1747174985562521.
[I 2025-03-20 01:08:16,922] Trial 45 finished with value: 0.17794141855284712 and parameters: {'C': 0.014155734607864626, 'penalty': 'l1'}. Best is trial 14 with value: 0.1747174985562521.
[I 2025-03-20 01:08:17,176] Trial 46 finished with value: 0.17516918886398464 and parameters: {'C': 0.022446316728720484, 'penalty': 'l2'}. Best is trial 14 with value: 0.1747174985562521.
[I 2025-03-20 01:08:17,287] Trial 47 finished with value: 0.25 and parameters: {'C': 0.0001977162340013098, 'penalty': 'l1'}. Best is trial 14 with value: 0.1747174985562521.
[I 2025-03-20 01:08:17,387] Trial 48 finished with value: 0.25 and parameters: {'C': 0.0007794849473181384, 'penalty': 'l1'}. Best is trial 14 with value: 0.1747174985562521.
[I 2025-03-20 01:08:17,735] Trial 49 finished with value: 0.1763431186869397 and par

[I 2025-03-20 01:08:29,112] Trial 88 finished with value: 0.1752853831478321 and parameters: {'C': 0.03659077178406955, 'penalty': 'l2'}. Best is trial 14 with value: 0.1747174985562521.
[I 2025-03-20 01:08:29,328] Trial 89 finished with value: 0.17481142334745314 and parameters: {'C': 0.05761124037575562, 'penalty': 'l1'}. Best is trial 14 with value: 0.1747174985562521.
[I 2025-03-20 01:08:29,501] Trial 90 finished with value: 0.17987144830411916 and parameters: {'C': 0.010800869151875647, 'penalty': 'l1'}. Best is trial 14 with value: 0.1747174985562521.
[I 2025-03-20 01:08:29,748] Trial 91 finished with value: 0.17472272934633876 and parameters: {'C': 0.07693765369366297, 'penalty': 'l1'}. Best is trial 14 with value: 0.1747174985562521.
[I 2025-03-20 01:08:30,003] Trial 92 finished with value: 0.17471837734719195 and parameters: {'C': 0.0783267046215316, 'penalty': 'l1'}. Best is trial 14 with value: 0.1747174985562521.
[I 2025-03-20 01:08:30,440] Trial 93 finished with value: 0.1

Best Logistic Regression CV Params (All): {'C': 0.07313397838472636, 'penalty': 'l1'}
Final Logistic Regression Brier Score (All): 0.16175026471786666


In [78]:
year_range = [i for i in range(2011, 2020)] + [i for i in range(2022, 2025)]
start_season = 2010 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 25 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.95, 1.05] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
# ----------------------------------------------------------
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
brier_women_list = []
brier_men_list = []
brier_all_list = []
for i in range(len(year_range)):
    df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
    df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
    df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
    df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
    
    x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
    x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
    x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
    x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])
    
    x_train_women_list.append(x_train_women)
    x_test_women_list.append(x_test_women)
    x_train_men_list.append(x_train_men)
    x_test_men_list.append(x_test_men)
    
    y_train_women_list.append(y_train_women)
    y_test_women_list.append(y_test_women)
    y_train_men_list.append(y_train_men)
    y_test_men_list.append(y_test_men)
    
# -------------------------------------------------------

for i in range(len(year_range)):
    print("Predictions for season ", year_range[i])
    x_train_women = x_train_women_list[i]
    y_train_women = y_train_women_list[i]
    x_test_women = x_test_women_list[i]
    y_test_women = y_test_women_list[i]
    
    best_lr_women.fit(x_train_women, y_train_women)
    y_prob_women = best_lr_women.predict_proba(x_test_women)[:, 1]

    score = brier_score_loss(y_test_women, y_prob_women)
    brier_women_list.append(score)
    print('LogisticRegressionCV for All trained on All', score)
    # ----------------------------------------------
#     x_train_men = x_train_men_list[i]
#     y_train_men = y_train_men_list[i]
#     x_test_men = x_test_men_list[i]
#     y_test_men = y_test_men_list[i]
    
#     best_lr_men.fit(x_train_men, y_train_men)
#     y_prob_men = best_lr_men.predict_proba(x_test_men)[:, 1]
    
#     score = brier_score_loss(y_test_men, y_prob_men)
#     brier_men_list.append(score)
#     print('LogisticRegressionCV for men trained on men', score)
    # ----------------------------------------------
#     y_prob = np.concatenate((y_prob_women, y_prob_men))
#     y_test = np.concatenate((y_test_women, y_test_men))
#     score = brier_score_loss(y_test, y_prob)
#     brier_all_list.append(score)
#     print('LogisticRegressionCV for all trained on women and men separately', score, '<---')
#     print()
    # ----------------------------------------------
print()
print('The mean score when trained on All and tested on All was: ', np.mean(brier_women_list), 
      'with a std of ', np.std(brier_women_list))
# print('The mean score when trained on men and tested on men was: ', np.mean(brier_men_list), 
#       'with a std of ', np.std(brier_men_list))
# print('The mean score when trained separately and tested on both was: ', np.mean(brier_all_list), 
#       'with a std of ', np.std(brier_all_list))

Predictions for season  2011
LogisticRegressionCV for All trained on All 0.16584764802798796
Predictions for season  2012
LogisticRegressionCV for All trained on All 0.14251803736858845
Predictions for season  2013
LogisticRegressionCV for All trained on All 0.15814860445358003
Predictions for season  2014
LogisticRegressionCV for All trained on All 0.14826528337199382
Predictions for season  2015
LogisticRegressionCV for All trained on All 0.1348007228522355
Predictions for season  2016
LogisticRegressionCV for All trained on All 0.17337859669300473
Predictions for season  2017
LogisticRegressionCV for All trained on All 0.14480184914512148
Predictions for season  2018
LogisticRegressionCV for All trained on All 0.15631222056382893
Predictions for season  2019
LogisticRegressionCV for All trained on All 0.1267699534834348
Predictions for season  2022
LogisticRegressionCV for All trained on All 0.16487939238537758
Predictions for season  2023
LogisticRegressionCV for All trained on All

# Selected columns

In [73]:
columns_to_include_women = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]
columns_to_include_men = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]

year_range = [2022]
start_season = 2010 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 25 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.95, 1.05] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
# -------------------------------------------
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
for i in range(len(year_range)):

    if len(year_range) > 1:
        df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
        df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
        df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
        df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
        
        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())
        
        x_train_women_list.append(x_train_women)
        x_test_women_list.append(x_test_women)
        x_train_men_list.append(x_train_men)
        x_test_men_list.append(x_test_men)

        y_train_women_list.append(y_train_women)
        y_test_women_list.append(y_test_women)
        y_train_men_list.append(y_train_men)
        y_test_men_list.append(y_test_men)
    
    else:
        df_train_women_list = df_train_women_list[0][columns_to_include_women]
        df_test_women_list = df_test_women_list[0][columns_to_include_women]
        df_train_men_list = df_train_men_list[0][columns_to_include_men]
        df_test_men_list = df_test_men_list[0][columns_to_include_men]

        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list)
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list)
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list)
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list)

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())

In [74]:
import optuna
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import StratifiedKFold

# Define an objective function that uses StratifiedKFold CV.
def objective_lr_cv(trial, X, y, n_folds=5):  # Increased folds for better generalization
    # Stronger regularization: Reduce range of C to prevent overfitting
    C = trial.suggest_float("C", 1e-4, 10, log=True)  # Lower upper bound

    solver_men = "liblinear" if best_params_men["penalty"] == "l1" else "lbfgs"
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    solver = solver_men  # More stable for both L1 and L2

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X, y):
        X_train_cv, X_val_cv = X[train_idx], X[val_idx]
        y_train_cv, y_val_cv = y[train_idx], y[val_idx]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(penalty=penalty, C=C, solver=solver, max_iter=3000))  # Increased max_iter
        ])

        model.fit(X_train_cv, y_train_cv)
        y_pred = model.predict_proba(X_val_cv)[:, 1]
        scores.append(brier_score_loss(y_val_cv, y_pred))

    return np.mean(scores)

###########################
# For Women
###########################
x_train_women = np.concatenate((x_train_women, x_train_men))
y_train_women = np.concatenate((y_train_women, y_train_men))

study_women = optuna.create_study(direction="minimize")
study_women.optimize(lambda trial: objective_lr_cv(trial, x_train_women, y_train_women), n_trials=100)  # Increased trials for better tuning

best_params_women = study_women.best_params
print("Best Logistic Regression CV Params (All):", best_params_women)

best_lr_women = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(penalty=best_params_women["penalty"],
                                       C=best_params_women["C"],
                                       solver="saga",
                                       max_iter=3000))
])
best_lr_women.fit(x_train_women, y_train_women)
y_pred_women_lr = best_lr_women.predict_proba(x_test_women)[:, 1]
brier_women_lr = brier_score_loss(y_test_women, y_pred_women_lr)
print("Final Logistic Regression Brier Score (All):", brier_women_lr)

[I 2025-03-20 01:05:30,703] A new study created in memory with name: no-name-b05e6483-2057-42a7-879e-1425a640a676
[I 2025-03-20 01:05:30,798] Trial 0 finished with value: 0.18128630655460737 and parameters: {'C': 0.0019810180883427977, 'penalty': 'l2'}. Best is trial 0 with value: 0.18128630655460737.
[I 2025-03-20 01:05:30,960] Trial 1 finished with value: 0.17539954314130277 and parameters: {'C': 0.03627810770295723, 'penalty': 'l1'}. Best is trial 1 with value: 0.17539954314130277.
[I 2025-03-20 01:05:31,053] Trial 2 finished with value: 0.25 and parameters: {'C': 0.00020361729359935393, 'penalty': 'l1'}. Best is trial 1 with value: 0.17539954314130277.
[I 2025-03-20 01:05:31,247] Trial 3 finished with value: 0.17551797467093838 and parameters: {'C': 0.35215696093994286, 'penalty': 'l2'}. Best is trial 1 with value: 0.17539954314130277.
[I 2025-03-20 01:05:31,334] Trial 4 finished with value: 0.19800858688556025 and parameters: {'C': 0.0005320744483933561, 'penalty': 'l2'}. Best is 

[I 2025-03-20 01:05:40,651] Trial 44 finished with value: 0.17529024179068364 and parameters: {'C': 0.18182285949257063, 'penalty': 'l2'}. Best is trial 25 with value: 0.1748956593151613.
[I 2025-03-20 01:05:40,829] Trial 45 finished with value: 0.1748995190080535 and parameters: {'C': 0.03637958203027574, 'penalty': 'l2'}. Best is trial 25 with value: 0.1748956593151613.
[I 2025-03-20 01:05:41,057] Trial 46 finished with value: 0.17492879891032223 and parameters: {'C': 0.05830787482853219, 'penalty': 'l2'}. Best is trial 25 with value: 0.1748956593151613.
[I 2025-03-20 01:05:41,176] Trial 47 finished with value: 0.17528214580448973 and parameters: {'C': 0.014339106211882962, 'penalty': 'l2'}. Best is trial 25 with value: 0.1748956593151613.
[I 2025-03-20 01:05:41,307] Trial 48 finished with value: 0.1769563047643571 and parameters: {'C': 0.004957097615044643, 'penalty': 'l2'}. Best is trial 25 with value: 0.1748956593151613.
[I 2025-03-20 01:05:41,524] Trial 49 finished with value: 0.

[I 2025-03-20 01:05:48,520] Trial 88 finished with value: 0.17560673965575718 and parameters: {'C': 0.010308389076946655, 'penalty': 'l2'}. Best is trial 59 with value: 0.17489502380440786.
[I 2025-03-20 01:05:48,709] Trial 89 finished with value: 0.17510935399489574 and parameters: {'C': 0.11057479629158358, 'penalty': 'l2'}. Best is trial 59 with value: 0.17489502380440786.
[I 2025-03-20 01:05:48,887] Trial 90 finished with value: 0.17500158738660104 and parameters: {'C': 0.02310540074095582, 'penalty': 'l2'}. Best is trial 59 with value: 0.17489502380440786.
[I 2025-03-20 01:05:49,038] Trial 91 finished with value: 0.1749145092101217 and parameters: {'C': 0.053331031569913215, 'penalty': 'l2'}. Best is trial 59 with value: 0.17489502380440786.
[I 2025-03-20 01:05:49,225] Trial 92 finished with value: 0.17489555257391604 and parameters: {'C': 0.03930569450097148, 'penalty': 'l2'}. Best is trial 59 with value: 0.17489502380440786.
[I 2025-03-20 01:05:49,425] Trial 93 finished with val

Best Logistic Regression CV Params (All): {'C': 0.04117361676500464, 'penalty': 'l2'}
Final Logistic Regression Brier Score (All): 0.16163611899899435


In [75]:
year_range = [i for i in range(2011, 2020)] + [i for i in range(2022, 2025)]
start_season = 2010 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 25 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.95, 1.05] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
# ----------------------------------------------------------
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
brier_women_list = []
brier_men_list = []
brier_all_list = []
for i in range(len(year_range)):
    df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
    df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
    df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
    df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
    
    x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
    x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
    x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
    x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])
    
    x_train_women_list.append(x_train_women)
    x_test_women_list.append(x_test_women)
    x_train_men_list.append(x_train_men)
    x_test_men_list.append(x_test_men)
    
    y_train_women_list.append(y_train_women)
    y_test_women_list.append(y_test_women)
    y_train_men_list.append(y_train_men)
    y_test_men_list.append(y_test_men)
    
# -------------------------------------------------------

for i in range(len(year_range)):
    print("Predictions for season ", year_range[i])
    x_train_women = x_train_women_list[i]
    y_train_women = y_train_women_list[i]
    x_test_women = x_test_women_list[i]
    y_test_women = y_test_women_list[i]
    
    best_lr_women.fit(x_train_women, y_train_women)
    y_prob_women = best_lr_women.predict_proba(x_test_women)[:, 1]

    score = brier_score_loss(y_test_women, y_prob_women)
    brier_women_list.append(score)
    print('LogisticRegressionCV for women trained on women', score)
    # ----------------------------------------------
#     x_train_men = x_train_men_list[i]
#     y_train_men = y_train_men_list[i]
#     x_test_men = x_test_men_list[i]
#     y_test_men = y_test_men_list[i]
    
#     best_lr_men.fit(x_train_men, y_train_men)
#     y_prob_men = best_lr_men.predict_proba(x_test_men)[:, 1]
    
#     score = brier_score_loss(y_test_men, y_prob_men)
#     brier_men_list.append(score)
#     print('LogisticRegressionCV for men trained on men', score)
    # ----------------------------------------------
#     y_prob = np.concatenate((y_prob_women, y_prob_men))
#     y_test = np.concatenate((y_test_women, y_test_men))
#     score = brier_score_loss(y_test, y_prob)
#     brier_all_list.append(score)
#     print('LogisticRegressionCV for all trained on women and men separately', score, '<---')
#     print()
    # ----------------------------------------------
print()
print('The mean score when trained on All and tested on All was: ', np.mean(brier_women_list), 
      'with a std of ', np.std(brier_women_list))
# print('The mean score when trained on men and tested on men was: ', np.mean(brier_men_list), 
#       'with a std of ', np.std(brier_men_list))
# print('The mean score when trained separately and tested on both was: ', np.mean(brier_all_list), 
#       'with a std of ', np.std(brier_all_list))

Predictions for season  2011
LogisticRegressionCV for women trained on women 0.1559821004146653
Predictions for season  2012
LogisticRegressionCV for women trained on women 0.13290394100148314
Predictions for season  2013
LogisticRegressionCV for women trained on women 0.17615009054401246
Predictions for season  2014
LogisticRegressionCV for women trained on women 0.1447394761526529
Predictions for season  2015
LogisticRegressionCV for women trained on women 0.14082691244505505
Predictions for season  2016
LogisticRegressionCV for women trained on women 0.1805437478350272
Predictions for season  2017
LogisticRegressionCV for women trained on women 0.14646573819602088
Predictions for season  2018
LogisticRegressionCV for women trained on women 0.15556835474816827
Predictions for season  2019
LogisticRegressionCV for women trained on women 0.13413754003310532
Predictions for season  2022
LogisticRegressionCV for women trained on women 0.1657854942716163
Predictions for season  2023
Logis